# NB10 · El recorrido de entrega, de principio a fin

El sistema completo con la configuración ya decidida: **preparar → ingerir → consultar → mutar → volver a consultar → evaluar → limpiar**. No se decide nada aquí ni se compara nada nuevo; es el recorrido que debe poder ejecutarse en un entorno limpio siguiendo el README.

**Antes de empezar:** `make motor-up MOTOR=qdrant`.

Trabaja sobre su **propia colección** (sufijo `__e2e`), así que arranca siempre desde cero y no toca el índice de NB04-NB09. La sección F la borra.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000) · consultas_evaluacion.csv (12)
#            · consultas_filtradas.csv (4) · eventos_catalogo.csv (24)
#            · control: consultas_desarrollo.csv (8) + relevancias_desarrollo.csv
import os
import sys
from functools import lru_cache
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path("..") / "src"))

from dotenv import load_dotenv

from aurum.almacen import PAYLOAD_SCHEMAS, add_normalized_key, build_payload
from aurum.busqueda import BuscadorVectorial, DenseRetriever, auditar_filtro_de_marca
from aurum.datos import load_csv
from aurum.embeddings import (
    GeminiEncoder, cache_key, corpus_fingerprint, encode_corpus, truncate_dim,
)
from aurum.evaluacion import evaluate_rankings, qrels_from_judgements
from aurum.motores import CATALOG_PREFIX, catalog_collection_name
from aurum.motores.aceptacion import self_retrieval_canaries
from aurum.motores.base import Point
from aurum.motores.qdrant import QdrantStore
from aurum.mutaciones import (
    aplicar_secuencia, clasificar_eventos, esperar_visibilidad, verificar_evento,
)
from aurum.plantillas import render_template

load_dotenv(Path("..") / ".env")
DATA = Path("..") / "data"
CACHE = Path("..") / "artifacts" / "embeddings"

# La configuracion ya decidida -config/config.yaml-, no se toca nada aqui.
MODELO, CONTRATO, PLANTILLA = "gemini-embedding-2", "sin_contrato", "A4"  # R02 · R01
DIM, TOP_K = 768, 10                  # D09b
EF = 32                               # R04
LOTE = 128                            # D15
ESQUEMA = "completo"                  # D13
POLITICA_NULOS = "cadena_vacia"       # D14
NORMALIZACION = "unaccent"            # D03
HNSW_M, HNSW_EF_CONSTRUCT = 16, 100   # D17
CAMPOS_FILTRABLES = ["brand", "color"]

# Coleccion propia: el recorrido se ejecuta entero sin tocar el indice de
# NB04-NB09, del que dependen los artefactos ya escritos.
COLECCION = catalog_collection_name(
    model=MODELO, template=PLANTILLA, dim=DIM,
) + "__e2e"

completo = load_csv(DATA / "catalogo_productos.csv")
evaluacion = load_csv(DATA / "consultas_evaluacion.csv")
filtradas = load_csv(DATA / "consultas_filtradas.csv")
eventos = clasificar_eventos(load_csv(DATA / "eventos_catalogo.csv"))

# Solo para la celda de control (seccion B): son las que calibraron el
# sistema, asi que no pueden ser la ejecucion principal.
desarrollo = load_csv(DATA / "consultas_desarrollo.csv")
QRELS = qrels_from_judgements(load_csv(DATA / "relevancias_desarrollo.csv"))

# Consultas escritas para esta demostracion: no salen de ningun CSV ni
# han intervenido en ninguna decision. Una por registro linguistico.
CONSULTAS_PROPIAS = {
    "PROPIA-1-literal": "auriculares inalambricos con cancelacion de ruido",
    "PROPIA-2-necesidad": "algo para tapar la ventana y que no entre luz por la manana",
    "PROPIA-3-vaga": "un regalo original para el amigo invisible",
}

print(f"coleccion : {COLECCION}")
print(f"catalogo  : {len(completo)} productos")
print(f"consultas : {len(evaluacion)} de evaluacion · {len(filtradas)} filtradas "
      f"· {len(CONSULTAS_PROPIAS)} propias")
print(f"control   : {len(desarrollo)} de desarrollo (seccion B)")
print(eventos["tipo"].value_counts().to_string())

c:\Users\asus\Master\modulos\modulo10_bbdd\practica\AURUM_MARKET\aurum-market-catalog\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


coleccion : aurum_catalogo__gemini_embedding_2__A4__768__e2e
catalogo  : 15000 productos
consultas : 12 de evaluacion · 4 filtradas · 3 propias
control   : 8 de desarrollo (seccion B)
tipo
actualizacion    8
baja             8
alta             8


## A · Ingerir el catálogo

Codifica los 15.000 productos de `catalogo_productos.csv` y los ingiere en Qdrant. La ingesta se lanza **dos veces** con los mismos puntos: el recuento debe ser el mismo.

Después espera —reintentando y contando el tiempo— a que el índice HNSW refleje la ingesta, antes de dejar que nadie consulte.

Los vectores salen de la caché; si no están, la celda para en vez de pagar 15.000 llamadas.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000) · ⚠️ requiere `make motor-up MOTOR=qdrant`
CORPUS_ID = f"catalogo_productos__{PLANTILLA}"
textos = render_template(completo, PLANTILLA)
clave = cache_key(
    model_id=MODELO, kind="document", contract=CONTRATO,
    corpus_id=CORPUS_ID, fingerprint=corpus_fingerprint(textos),
)
if not (CACHE / f"{clave}.npy").exists():
    raise RuntimeError(
        f"Los vectores de {CORPUS_ID} no estan en cache ({clave}).\n"
        f"Esta celda no paga 15.000 llamadas nuevas."
    )

encoder = GeminiEncoder(
    api_key=os.environ.get("GEMINI_API_KEY"), model_id=MODELO,
    native_dim=3072, window=8192,
)
vectores = truncate_dim(
    encode_corpus(
        encoder, textos, corpus_id=CORPUS_ID, kind="document",
        contract=CONTRATO, batch_size=32, cache_dir=CACHE,
    ).vectors,
    DIM,
)

con_claves = completo.copy()
for campo in CAMPOS_FILTRABLES:
    con_claves = add_normalized_key(con_claves, field=campo, mode=NORMALIZACION)

puntos = [
    Point(
        record_id=fila["record_id"],
        vector=vectores[i],
        payload=build_payload(
            fila, fields=PAYLOAD_SCHEMAS[ESQUEMA], null_policy=POLITICA_NULOS
        ),
    )
    for i, fila in enumerate(con_claves.to_dict("records"))
]

indice = QdrantStore(
    collection=COLECCION,
    url=os.environ.get("AURUM_QDRANT_URL", "http://localhost:6333"),
    api_key=os.environ.get("AURUM_QDRANT_API_KEY"),
    prefix=CATALOG_PREFIX,
    timeout=60,
)
indice.create_collection(
    dim=DIM, metric="cosine",
    hnsw_m=HNSW_M, hnsw_ef_construct=HNSW_EF_CONSTRUCT,
)

indice.upsert(puntos, batch_size=LOTE)
RECUENTO_1 = indice.count()
indice.upsert(puntos, batch_size=LOTE)          # la misma ingesta, otra vez
RECUENTO_2 = indice.count()

# D18: espera activa, no un sleep a ciegas. Y no basta con mirar el estado:
# `upsert(wait=True)` garantiza que el punto esta escrito, no que la
# coleccion entera este servible. Si aqui se deja pasar una ingesta a
# medias, las secciones de abajo miden sobre un catalogo incompleto y el
# sintoma es sutil -devuelven algo, solo que peor y con menos similitud-.
# Por eso la condicion son tres cosas a la vez.
canarios = self_retrieval_canaries(puntos, n=3)
ESPERADOS = len(puntos)


def indice_servible():
    """1) estan todos los puntos, 2) Qdrant dice verde y 3) responde.

    El canario se busca con su PROPIO vector, asi que la respuesta
    correcta no depende de la calidad del modelo: debe volver el, el
    primero. Si vuelve otro, lo roto es el indice."""
    if indice.count() != ESPERADOS or not indice.index_ready():
        return False
    return all(
        (lambda hits: bool(hits) and hits[0].record_id == punto.record_id)(
            indice.search(punto.vector, top_k=1, ef=EF)
        )
        for punto in canarios
    )


indexado = esperar_visibilidad(indice_servible, timeout_s=180.0, intervalo_s=1.0)

if not indexado["visible"]:
    raise RuntimeError(
        f"El indice no quedo servible en {indexado['segundos']:.0f} s: "
        f"{indice.count()} de {ESPERADOS} puntos · verde={indice.index_ready()}.\n"
        f"Seguir mediria sobre un catalogo incompleto."
    )

tabla_ingesta = pd.DataFrame([
    {"comprobacion": "puntos tras la 1a ingesta", "valor": RECUENTO_1},
    {"comprobacion": "puntos tras la 2a ingesta", "valor": RECUENTO_2},
    {"comprobacion": "puntos esperados", "valor": ESPERADOS},
    {"comprobacion": "segundos hasta servible", "valor": round(indexado["segundos"], 1)},
    {"comprobacion": "comprobaciones hasta servible", "valor": indexado["intentos"]},
])
tabla_ingesta.style.hide(axis="index")

comprobacion,valor
puntos tras la 1a ingesta,15000.000000
puntos tras la 2a ingesta,15000.000000
puntos esperados,15000.000000
segundos hasta servible,1.100000
comprobaciones hasta servible,2.000000


## B · Control: ¿responde el motor?

Comprobación previa, no la ejecución principal. Lanza las 8 consultas de `consultas_desarrollo.csv` —las únicas con juicios de relevancia— y las 4 de `consultas_filtradas.csv`, que llevan filtro de marca.

Son las consultas con las que se calibró el sistema, así que sus métricas confirman que el índice está bien construido, no que el sistema sea bueno.

In [ ]:
# 📄 DATOS · consultas_desarrollo.csv (8) + consultas_filtradas.csv (4)
@lru_cache(maxsize=256)
def codificar_consulta(texto: str):
    codificado = encode_corpus(
        encoder, [texto], corpus_id="consulta_suelta", kind="query",
        contract=CONTRATO, batch_size=1, cache_dir=CACHE,
    )
    return truncate_dim(codificado.vectors, DIM)[0]


buscador = BuscadorVectorial(indice, codificar_consulta, top_k=TOP_K, ef=EF)

control = {
    str(qid): [r.document_id for r in buscador.buscar(texto, top_k=TOP_K)]
    for qid, texto in zip(desarrollo["query_id"], desarrollo["query_text"])
}
METRICAS_CONTROL = evaluate_rankings(control, QRELS, k=TOP_K).summary

alcance_por_marca = completo["brand"].value_counts().to_dict()
tabla_filtros = auditar_filtro_de_marca(
    buscador, filtradas.to_dict("records"), alcance=alcance_por_marca, top_k=TOP_K,
)
FILTROS_OK = int(
    tabla_filtros["veredicto"].str.startswith("✅").sum()
)
print(f"consultas filtradas puras: {FILTROS_OK}/{len(tabla_filtros)}")
display(
    pd.DataFrame([METRICAS_CONTROL])
    .style.hide(axis="index")
    .format("{:.4f}")
)
tabla_filtros.style.hide(axis="index")

consultas filtradas puras: 4/4


precision_at_10,recall_at_10,mrr_at_10,ndcg_at_10
0.6500,0.3357,0.9375,0.6235


caso,consulta,marca,n_en_catalogo,n_resultados,de_la_marca,pureza,veredicto
FILTER-001,herramienta inalámbrica para perforar,Einhell,30,10,10,100%,✅ pureza 100 % y cobertura completa (10 de 10)
FILTER-002,tableta ligera para estudiar y tomar apuntes,Apple,100,10,10,100%,✅ pureza 100 % y cobertura completa (10 de 10)
FILTER-003,zapatillas cómodas para salir a correr,NIKE,295,10,10,100%,✅ pureza 100 % y cobertura completa (10 de 10)
FILTER-004,monitor para trabajar con varias ventanas,SAMSUNG,155,10,10,100%,✅ pureza 100 % y cobertura completa (10 de 10)


## C · La ejecución real: consultas de evaluación

Las 12 de `consultas_evaluacion.csv` —4 intenciones escritas de tres formas cada una— más 3 consultas escritas para esta demostración, que no salen de ningún fichero.

Ninguna tiene juicios de relevancia, así que aquí no hay nDCG: se mira lo que devuelve y, en la sección F, cuánto se mueve al cambiar el catálogo.

Este es el camino **sin modificaciones**.

In [ ]:
# 📄 DATOS · consultas_evaluacion.csv (12) + 3 consultas propias
CONSULTAS = {
    **dict(zip(evaluacion["evaluation_id"], evaluacion["query_text"])),
    **CONSULTAS_PROPIAS,
}

RESULTADOS_ANTES = {
    cid: buscador.buscar(texto, top_k=TOP_K) for cid, texto in CONSULTAS.items()
}
ANTES = {cid: [r.document_id for r in res] for cid, res in RESULTADOS_ANTES.items()}


def mostrar(resultados, consulta, *, etiqueta="", n=3, campo=None):
    """Lo que se envia y lo que se recibe, uno encima del otro.

    Sin juicios de relevancia no hay metrica que calcular aqui: lo unico
    que se puede hacer es leer los titulos y ver si tienen sentido."""
    print(f"┌─ {etiqueta}" if etiqueta else "┌─")
    print(f'│  ENVIO   "{consulta}"')
    print("│  RECIBO")
    if not resultados:
        print("│    (sin resultados)")
    for r in resultados[:n]:
        extra = f" · {campo}={r.metadatos.get(campo, '')!r}" if campo else ""
        print(f"│    {r.rank}. [{r.score:.3f}] {r.titulo[:58]}")
        print(f"│       {r.document_id}{extra}")
    print("└─\n")


# Las tres propias: tres registros distintos de la misma clase de cliente.
for cid, texto in CONSULTAS_PROPIAS.items():
    mostrar(RESULTADOS_ANTES[cid], texto, etiqueta=cid)

# Y una intencion del CSV en sus tres formulaciones: misma necesidad,
# tres maneras de escribirla.
INTENCION_MUESTRA = str(evaluacion["evaluation_id"].iloc[0]).split("-")[1]
for cid in [c for c in evaluacion["evaluation_id"] if INTENCION_MUESTRA in c]:
    mostrar(RESULTADOS_ANTES[cid], CONSULTAS[cid], etiqueta=cid)

print(f"consultas lanzadas: {len(ANTES)} · "
      f"resultados por consulta: {min(len(v) for v in ANTES.values())}"
      f"-{max(len(v) for v in ANTES.values())}")

┌─ PROPIA-1-literal
│  ENVIO   "auriculares inalambricos con cancelacion de ruido"
│  RECIBO
│    1. [0.673] Nuomaidi Auriculares Bluetooth Deportivos,Auriculares con 
│       B07PJMCTHF
│    2. [0.664] Auriculares Bluetooth, Auriculares Inalámbricos Bluetooth 
│       B07Z2GGRMB
│    3. [0.653] Auriculares Bluetooth,Auriculares Bluetooth Inalámbricos M
│       B07NZW27SB
└─

┌─ PROPIA-2-necesidad
│  ENVIO   "algo para tapar la ventana y que no entre luz por la manana"
│  RECIBO
│    1. [0.593] BBestseller 1 Piezas Sombra Cortina de Copo de Nieve Visil
│       B07R8ZB547
│    2. [0.591] Sello de Ventanas Impermeable Nueva telescópica portable f
│       B09D99G6T2
│    3. [0.579] Tequial - Melatonina 1 Mg, 60 Comprimidos Sublinguales
│       B01K8559QQ
└─

┌─ PROPIA-3-vaga
│  ENVIO   "un regalo original para el amigo invisible"
│  RECIBO
│    1. [0.600] Tarjeta Regalo Amazon.es - Tarjeta Desplegable Regalo Navi
│       B07T7JCYX6
│    2. [0.596] Diario para familias modernas: Juegos, ac

### C.1 · El filtro por color

La misma consulta, sin filtro y filtrando por color, sobre el catálogo recién ingerido. El filtro lo ejecuta Qdrant contra el índice de texto de `color_normalized`; no se descarta nada en Python.

Va por el almacén, no por `buscar()`: NB05 decidió no exponer el color en la interfaz pública, así que esto enseña la capacidad del índice sin abrir una puerta nueva.

El filtro casa **palabras** y no subcadenas: `rosa` no arrastra `rosado`. Fue una de las razones de elegir este motor.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv · el color va en el payload (D13/D14)
from aurum.datos import normalize_brand
from aurum.motores.base import FilterCondition

CONSULTA_COLOR = "vestido de fiesta para una boda"
COLORES = ["negro", "rojo", "azul", "rosa"]
vector_color = codificar_consulta(CONSULTA_COLOR)


def buscar_por_color(color=None, k=TOP_K):
    """`contains` sobre color_normalized. D03: se filtra por la clave
    derivada, y el valor pedido se normaliza igual que el almacenado.

    Va por el almacen y no por `buscador.buscar()` a proposito: NB05
    decidio no exponer el color en la interfaz publica, asi que esto es
    la capacidad del indice, no una puerta de entrada nueva."""
    filtros = ()
    if color is not None:
        pedido = normalize_brand(color, NORMALIZACION)
        # D14: con los nulos como cadena vacia, un `contains ""` casa con
        # TODO y el filtro deja de filtrar sin avisar. Un color en blanco
        # es entrada invalida, no un filtro; para no filtrar, color=None.
        if not pedido:
            raise ValueError(
                f"{color!r} se normaliza a vacio: seria un filtro que no filtra. "
                f"Para buscar sin filtrar, pasa color=None."
            )
        filtros = (FilterCondition(
            field="color", value=pedido, operator="contains",
        ),)
    return indice.search(vector_color, top_k=k, filters=filtros, ef=EF)


def como_resultados(hits):
    """Los SearchHit del motor, con la forma que imprime `mostrar`."""
    from aurum.busqueda import Resultado
    return [
        Resultado(
            document_id=str(h.payload.get("product_id", "")),
            rank=h.rank, score=h.score,
            record_id=h.record_id,
            titulo=str(h.payload.get("title", "")),
            metadatos=h.payload,
        )
        for h in hits
    ]


mostrar(como_resultados(buscar_por_color()), CONSULTA_COLOR,
        etiqueta="sin filtro", campo="color")
for color in COLORES:
    mostrar(como_resultados(buscar_por_color(color)), CONSULTA_COLOR,
            etiqueta=f"filtrando color={color!r}", campo="color")

# Pureza: de lo devuelto, cuanto lleva de verdad el color pedido.
# Se tokeniza con \w+ y no con .split(): el indice de texto de Qdrant usa
# el tokenizador WORD, que parte tambien por `/`, `-` y `.` -asi
# "blanco/rosa." es rosa para el motor-. Con .split() saldrian coladas
# que no lo son.
import re

colores_catalogo = con_claves["color_normalized"].fillna("")


def lleva_el_color(texto, pedido):
    return pedido in re.findall(r"\w+", str(texto))


filas_color = []
for color in COLORES:
    pedido = normalize_brand(color, NORMALIZACION)
    hits = buscar_por_color(color)
    filas_color.append({
        "color_pedido": color,
        "en_el_catalogo": int(
            colores_catalogo.apply(lleva_el_color, pedido=pedido).sum()
        ),
        "devueltos": len(hits),
        "de_ese_color": sum(
            lleva_el_color(h.payload.get("color_normalized", ""), pedido)
            for h in hits
        ),
    })
pd.DataFrame(filas_color).style.hide(axis="index")

┌─ sin filtro
│  ENVIO   "vestido de fiesta para una boda"
│  RECIBO
│    1. [0.601] Vectry Vestidos Adolescentes Chica Vestidos Casuales Vesti
│       B07MDTWD7X · color='Amarillo'
│    2. [0.571] Vestidos Largo Mujer de Novia POLP Elegantes Tallas Grande
│       B07L61T75Z · color='Blanco'
│    3. [0.558] Fitness
│       8425518261 · color=''
└─

┌─ filtrando color='negro'
│  ENVIO   "vestido de fiesta para una boda"
│  RECIBO
│    1. [0.545] Kanlin1986 Vestido Largo De Navidad para Mujer, Vestido Ta
│       B0818K237B · color='Negro'
│    2. [0.543] Vectry Faldas Falda De Flamenca Niña Faldas Mujer Cortas F
│       B081GVFHDF · color='Negro'
│    3. [0.537] MUJER FELIZ Disfraz Costume Niña, Princesa Disfraz Vestido
│       B07TFKMTJ8 · color='Negro'
└─

┌─ filtrando color='rojo'
│  ENVIO   "vestido de fiesta para una boda"
│  RECIBO
│    1. [0.536] K-youth® Floral Vestidos de Fiesta Largos De Noche Sin Man
│       B077T8RWB4 · color='Rojo'
│    2. [0.477] Kanlin1986 Mujer Retro Navi

color_pedido,en_el_catalogo,devueltos,de_ese_color
negro,2202,10,10
rojo,371,10,10
azul,633,10,10
rosa,252,10,10


### C.2 · Qué se pierde por aproximar

Las mismas tres consultas propias contra el índice con distintos valores de `ef`, comparadas con la búsqueda exacta sobre los mismos vectores. El sistema entregado usa `ef=32` (R04).

`recall_ann@10` es la fracción de los 10 vecinos exactos que el índice recupera: **más alto es mejor**, 1,0 es fidelidad total. No mide si el resultado es bueno, mide si el índice reproduce lo que el modelo dice — son dos preguntas distintas.

In [ ]:
# 📄 DATOS · las 3 consultas propias · el oraculo son los mismos vectores
# de la seccion A, buscados sin aproximar (recorre los 15.000, ~46 MB).
oraculo = DenseRetriever(vectores, completo["product_id"].tolist(), metric="cosine")
EFS = (32, 64, 128, 256)

filas_fidelidad, top1 = [], {}
for cid, texto in CONSULTAS_PROPIAS.items():
    qv = codificar_consulta(texto)
    exactos = [r.document_id for r in oraculo.search_vector(qv, k=TOP_K)]
    fila = {"consulta": cid}
    for ef in EFS:
        hits = indice.search(qv, top_k=TOP_K, ef=ef)
        ids = [str(h.payload.get("product_id")) for h in hits]
        fila[f"recall_ann@10_ef{ef}"] = len(set(ids) & set(exactos)) / TOP_K
        top1[(cid, ef)] = hits[0]
    filas_fidelidad.append(fila)

tabla_fidelidad = pd.DataFrame(filas_fidelidad)

# La consulta que mas se degrada, con su top-1 en cada extremo: la cifra
# sola no ensena que producto se pierde por el camino.
columna_baja, columna_alta = f"recall_ann@10_ef{EFS[0]}", f"recall_ann@10_ef{EFS[-1]}"
peor = tabla_fidelidad.loc[tabla_fidelidad[columna_baja].idxmin(), "consulta"]
print(f'{peor} · "{CONSULTAS_PROPIAS[peor]}"')
for ef in (EFS[0], EFS[-1]):
    h = top1[(peor, ef)]
    print(f"  ef={ef:<4} [{h.score:.4f}] {str(h.payload.get('title'))[:56]}")
tabla_fidelidad.style.hide(axis="index").format(
    {c: "{:.1f}" for c in tabla_fidelidad.columns if c != "consulta"}
)

PROPIA-2-necesidad · "algo para tapar la ventana y que no entre luz por la manana"
  ef=32   [0.5925] BBestseller 1 Piezas Sombra Cortina de Copo de Nieve Vis
  ef=256  [0.6042] DRFQSK Cortinas Opacas Térmicas Aislantes para Salon Cor


consulta,recall_ann@10_ef32,recall_ann@10_ef64,recall_ann@10_ef128,recall_ann@10_ef256
PROPIA-1-literal,0.9,0.9,0.9,1.0
PROPIA-2-necesidad,0.3,0.3,1.0,1.0
PROPIA-3-vaga,0.9,0.9,1.0,1.0


### Cómo leer esta tabla

No es un fallo nuevo: es el coste de R04 hecho visible. NB06 ya registró `recall_ann_at_10_min = 0,0` y documentó la consulta 18868 con el mismo umbral de recuperación, `ef=128`.

Lo que añaden estas tres es que aquel caso duro **no era una rareza del conjunto de desarrollo**: son consultas escritas a mano, que no intervinieron en ninguna decisión, y aun así se degradan con `ef=32` y convergen en el mismo `ef=128`.

El problema no es el valor de `ef`, sino el criterio que lo eligió: *"menor p95 entre las admisibles"* optimiza la latencia agregada mientras el recall mínimo por consulta cae. Un criterio sobre el peor caso habría elegido `ef=128`, cuyo p95 —12,36 ms en `benchmark_ann.csv`— cabe en el presupuesto de 20 ms de D16. **R04 no se reabre aquí**; queda registrado en `config.yaml` como mejora medida.

## D · Aplicar el ciclo de vida del catálogo

Aplica los 24 eventos de `eventos_catalogo.csv` —8 altas, 8 modificaciones y 8 bajas— sobre la colección, y comprueba uno de cada tipo por lectura directa y por búsqueda.

In [ ]:
# 📄 DATOS · eventos_catalogo.csv (24) · ⚠️ escribe en Qdrant
eventos_upsert = eventos[eventos["tipo"] != "baja"].reset_index(drop=True)
eventos_baja = eventos[eventos["tipo"] == "baja"].reset_index(drop=True)

vectores_upsert = truncate_dim(
    encode_corpus(
        encoder, eventos_upsert["text"].tolist(),
        corpus_id="eventos_catalogo_upsert", kind="document",
        contract=CONTRATO, batch_size=16, cache_dir=CACHE,
    ).vectors,
    DIM,
)
vector_por_record = dict(zip(eventos_upsert["record_id"], vectores_upsert))

upsert_con_claves = eventos_upsert.copy()
for campo in CAMPOS_FILTRABLES:
    upsert_con_claves = add_normalized_key(
        upsert_con_claves, field=campo, mode=NORMALIZACION
    )
puntos_upsert = [
    Point(
        record_id=fila["record_id"],
        vector=vector_por_record[fila["record_id"]],
        payload=build_payload(
            fila, fields=PAYLOAD_SCHEMAS[ESQUEMA], null_policy=POLITICA_NULOS
        ),
    )
    for fila in upsert_con_claves.to_dict("records")
]

aplicar_secuencia(
    indice, puntos_upsert, eventos_baja["record_id"].tolist(), batch_size=LOTE,
)
RECUENTO_MUTADO = indice.count()

# Uno de cada tipo, por lectura directa y por busqueda vectorial (D18).
filas_visibilidad = []
for tipo in ("alta", "actualizacion", "baja"):
    fila = eventos[eventos["tipo"] == tipo].iloc[0]
    rid = fila["record_id"]
    traza = verificar_evento(
        indice, tipo, record_id=rid,
        vector=vector_por_record.get(rid),
        catalog_version_esperado=2 if tipo == "actualizacion" else None,
        top_k=5,
    )
    filas_visibilidad.append({
        "tipo": tipo,
        "product_id": fila["product_id"],
        "visible_por_id": traza["por_id"]["visible"],
        "visible_por_busqueda": traza["por_busqueda"]["visible"],
    })

print(f"puntos antes de los eventos  : {RECUENTO_2}")
print(f"puntos despues de los eventos: {RECUENTO_MUTADO}"
      f"  (-{len(eventos_baja)} bajas +{int((eventos['tipo'] == 'alta').sum())} altas)")
pd.DataFrame(filas_visibilidad).style.hide(axis="index")

puntos antes de los eventos  : 15000
puntos despues de los eventos: 15000  (-8 bajas +8 altas)


tipo,product_id,visible_por_id,visible_por_busqueda
alta,AURUM-NEW-001,True,True
actualizacion,B000G3T55M,True,True
baja,B081JP8CC6,True,None


## E · Volver a consultar, ya con el catálogo mutado

Repite exactamente las 15 consultas de la sección C sobre la colección modificada.

Este es el camino **con modificaciones**.

In [ ]:
# 📄 DATOS · las mismas 15 consultas de la seccion C
DESPUES = {
    cid: [r.document_id for r in buscador.buscar(texto, top_k=TOP_K)]
    for cid, texto in CONSULTAS.items()
}

tabla_cambios = pd.DataFrame([
    {
        "consulta": cid,
        "cambia_el_top10": ANTES[cid] != DESPUES[cid],
        "productos_que_entran": len([p for p in DESPUES[cid] if p not in ANTES[cid]]),
    }
    for cid in CONSULTAS
])
print(f"consultas cuyo top-{TOP_K} cambia: "
      f"{int(tabla_cambios['cambia_el_top10'].sum())}/{len(tabla_cambios)}")
tabla_cambios.style.hide(axis="index")

consultas cuyo top-10 cambia: 13/15


consulta,cambia_el_top10,productos_que_entran
EVAL-100455-context,True,1
EVAL-100455-direct,True,1
EVAL-100455-semantic,True,1
EVAL-101352-context,True,1
EVAL-101352-direct,True,1
EVAL-101352-semantic,True,1
EVAL-93437-context,True,1
EVAL-93437-direct,True,1
EVAL-93437-semantic,True,1
EVAL-96202-context,True,1


## F · Los dos caminos, comparados

Sin juicios de relevancia no hay nDCG, así que se mide **Jaccard@10** entre lo que devolvió cada consulta antes y después de los eventos: 1,0 significa que el catálogo cambió pero esa consulta no se enteró; 0,0, que devuelve productos completamente distintos.

La segunda tabla mira otra cosa: si las tres formulaciones de una misma intención siguen coincidiendo entre sí después de mutar el catálogo.

In [ ]:
from aurum.evaluacion import formulation_consistency, jaccard_at_k

filas = []
for cid, texto in CONSULTAS.items():
    entran = [p for p in DESPUES[cid] if p not in ANTES[cid]]
    salen = [p for p in ANTES[cid] if p not in DESPUES[cid]]
    filas.append({
        "consulta": cid,
        "texto": texto[:44],
        "jaccard_antes_vs_despues": round(
            jaccard_at_k(ANTES[cid], DESPUES[cid], k=TOP_K), 4
        ),
        "entran_en_el_top10": ", ".join(entran) or "-",
        "salen_del_top10": ", ".join(salen) or "-",
    })
movimiento = pd.DataFrame(filas)

print(f"jaccard medio antes-vs-despues: "
      f"{movimiento['jaccard_antes_vs_despues'].mean():.4f}  "
      f"(1,0 = el catalogo cambio pero la consulta no se entero)")
print(f"consultas intactas: "
      f"{int((movimiento['jaccard_antes_vs_despues'] == 1.0).sum())}"
      f"/{len(movimiento)}")
movimiento.style.hide(axis="index")

jaccard medio antes-vs-despues: 0.8424  (1,0 = el catalogo cambio pero la consulta no se entero)
consultas intactas: 2/15


consulta,texto,jaccard_antes_vs_despues,entran_en_el_top10,salen_del_top10
EVAL-100455-context,taladro sin cable de 24 voltios que venga co,0.818200,AURUM-NEW-001,B01AY8BVQY
EVAL-100455-direct,taladro 24v batería,0.818200,AURUM-NEW-001,B01A5VQHBY
EVAL-100455-semantic,quiero una herramienta inalámbrica potente p,0.818200,AURUM-NEW-001,B01N6Y6G16
EVAL-101352-context,"tele de tamaño reducido para una cocina, alr",0.818200,AURUM-NEW-004,B08TLXPSPB
EVAL-101352-direct,television 28 pulgadas,0.818200,AURUM-NEW-004,B07XVL338B
EVAL-101352-semantic,busco un televisor pequeño de unas setenta c,0.818200,AURUM-NEW-004,B07CKWJK2B
EVAL-93437-context,me duele la espalda al trabajar y necesito u,0.818200,AURUM-NEW-002,B07KZXHXQ4
EVAL-93437-direct,sillas oficina ergonomicas,0.818200,AURUM-NEW-002,B08THD1V1K
EVAL-93437-semantic,necesito un asiento cómodo para trabajar och,0.818200,AURUM-NEW-002,B01H050GVU
EVAL-96202-context,pieza para sujetar un aire acondicionado en,0.818200,AURUM-NEW-003,B08742951P


In [ ]:
# Consistencia entre las tres formulaciones de cada intencion, antes y
# despues. Solo las 12 de evaluacion: las propias no van por intenciones.
ids_evaluacion = list(evaluacion["evaluation_id"])
consistencia = (
    formulation_consistency({c: ANTES[c] for c in ids_evaluacion}, k=TOP_K)
    .merge(
        formulation_consistency({c: DESPUES[c] for c in ids_evaluacion}, k=TOP_K),
        on="intencion", suffixes=("_antes", "_despues"),
    )
)
consistencia.style.hide(axis="index")

intencion,jaccard_context_direct_antes,jaccard_context_semantic_antes,jaccard_direct_semantic_antes,jaccard_context_direct_despues,jaccard_context_semantic_despues,jaccard_direct_semantic_despues
100455,0.818200,0.250000,0.250000,0.666700,0.333300,0.250000
101352,0.428600,0.250000,0.250000,0.538500,0.333300,0.333300
93437,0.052600,0.428600,0.111100,0.111100,0.428600,0.176500
96202,0.333300,0.666700,0.333300,0.333300,0.666700,0.333300


### Cómo leer estos números

Un Jaccard alto entre antes y después es lo esperable: 8 altas y 8 bajas sobre 15.000 productos solo deberían mover las consultas de su categoría. Una consulta que cambia entera señala que alguna alta compite directamente con ella.

La consistencia entre formulaciones no debería moverse apenas: mide si el sistema entiende la intención, y eso no depende de que el catálogo tenga ocho productos más.

## G · Limpiar

Borra la colección creada en la sección A. **No se ejecuta sola**: hay que poner `LIMPIAR = True`.

El índice de NB04-NB09 no se toca en ningún caso. Para parar el motor, `make motor-down MOTOR=qdrant`.

In [ ]:
LIMPIAR = False   # ponlo a True para borrar la coleccion de este notebook

if LIMPIAR:
    # Doble cerrojo: `recreate=True` mas AURUM_ALLOW_RESET en el entorno.
    indice.create_collection(dim=DIM, metric="cosine", recreate=True)
    print(f"{COLECCION} recreada vacia: {indice.count()} puntos")
else:
    print(f"{COLECCION} conservada con {indice.count()} puntos.")
    print("Pon LIMPIAR = True para borrarla · `make motor-down MOTOR=qdrant` para parar el motor.")

aurum_catalogo__gemini_embedding_2__A4__768__e2e recreada vacia: 0 puntos
